In [1]:
# 09_deploy_quantize.ipynb — Deploy & Quantize (full self-contained cell)
# Paste and run. This will:
#  - Inspect distillation outputs for each language
#  - Attempt TorchScript export (fallback ok)
#  - Do PyTorch dynamic quantization (for Linear/Embedding/LSTM/Transformer compatible parts)
#  - Optionally export ONNX (best-effort)
#  - Save tokenizer, quantized model state, and a metrics.json with sizes & latency
#  - Run a sanity inference test and compare logits/probs (pre/post)
#  - Save everything under outputs/deploy_quantize/<Lang>/<Model>/

import os, json, time, shutil, math, tempfile
from pathlib import Path
from typing import Dict, Any
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pathlib import Path
from tqdm import tqdm

# ---------- CONFIG ----------
ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2")
DISTILL_ROOT = ROOT / "outputs" / "distillation"
DEPLOY_ROOT = ROOT / "outputs" / "deploy_quantize"
STUDENT_MODEL_ID = "distil_distilbert-base-multilingual-cased"   # matches earlier notebooks
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_LATENCY_RUNS = 30
BATCH_SIZE_FOR_SANITY = 4
MAX_TEXT_SANITY = 512  # chars to use in sanity test texts (we do not truncate full dataset; only for quick sanity)
ONNX_OPSET = 13

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True); return p

def sizeof(path: Path):
    if not path.exists(): return 0
    if path.is_file(): return path.stat().st_size
    s=0
    for f in path.rglob("*"):
        if f.is_file(): s+=f.stat().st_size
    return s

def human_bytes(nbytes):
    if nbytes is None: return "N/A"
    for unit in ['B','KB','MB','GB','TB']:
        if nbytes < 1024.0:
            return f"{nbytes:3.2f}{unit}"
        nbytes /= 1024.0
    return f"{nbytes:.2f}PB"

def try_torchscript_export(model: torch.nn.Module, example_inputs: Dict[str, torch.Tensor], save_path: Path):
    """
    Attempt to export a torch.jit.trace'd module.
    Returns (ok:bool, message:str)
    """
    try:
        model_cpu = model.to("cpu").eval()
        # Trace expects positional args; supply input_ids and attention_mask as tuple
        # Use torch.jit.trace with example inputs packed as a tuple
        inputs_tuple = (example_inputs['input_ids'], example_inputs['attention_mask'])
        traced = torch.jit.trace(model_cpu, inputs_tuple, strict=False)
        traced.save(str(save_path))
        return True, "torchscript_trace_success"
    except Exception as e:
        return False, f"torchscript_failed: {repr(e)}"

def try_onnx_export(model: torch.nn.Module, tokenizer, example_texts, save_path: Path, opset=ONNX_OPSET):
    """
    Best-effort ONNX export for the sequence classification forward pass.
    Returns (ok:bool, message:str)
    """
    try:
        model_cpu = model.to("cpu").eval()
        enc = tokenizer(example_texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
        input_ids = enc['input_ids']
        attention_mask = enc['attention_mask']
        input_names = ['input_ids', 'attention_mask']
        output_names = ['logits']
        dynamic_axes = {'input_ids': {0:'batch',1:'seq'}, 'attention_mask': {0:'batch',1:'seq'}, 'logits': {0:'batch'}}
        torch.onnx.export(
            model_cpu,
            (input_ids, attention_mask),
            str(save_path),
            input_names=input_names,
            output_names=output_names,
            dynamic_axes=dynamic_axes,
            opset_version=opset,
            do_constant_folding=True
        )
        return True, "onnx_export_success"
    except Exception as e:
        return False, f"onnx_failed: {repr(e)}"

# For dynamic quantization (works on CPU)
def quantize_dynamic_model(orig_model: torch.nn.Module, mapping=None):
    """
    Applies torch.quantization.quantize_dynamic to the model.
    mapping: list of layer types to quantize (default Linear+LSTM+GRU)
    Returns quantized model on CPU.
    """
    q_types = mapping or {torch.nn.Linear, torch.nn.LSTM, torch.nn.GRU}
    try:
        qmodel = torch.quantization.quantize_dynamic(orig_model.cpu(), q_types, dtype=torch.qint8)
        return True, qmodel
    except Exception as e:
        return False, f"quantize_dynamic_failed: {repr(e)}"

# ---------- main ----------
ensure_dir(DEPLOY_ROOT)

languages = sorted([p.name for p in (ROOT/"data_splits").iterdir() if p.is_dir()])
print("Languages detected:", languages)

deploy_report = {}

for LANG in languages:
    print("\n" + "="*80)
    print(f"[LANG] {LANG}")
    distill_dir = DISTILL_ROOT / LANG / STUDENT_MODEL_ID
    if not distill_dir.exists():
        print(f"[WARN] Distillation output missing for {LANG} at {distill_dir} -> skipping")
        continue

    best_model_dir = distill_dir / "best_model"
    if not best_model_dir.exists():
        # sometimes model saved in run root; try direct folder
        alt = distill_dir / "best_model"
        if alt.exists(): best_model_dir = alt
        else:
            print(f"[ERROR] Best model dir missing for {LANG}. Expected: {best_model_dir}")
            continue

    out_dir = ensure_dir(DEPLOY_ROOT / LANG / STUDENT_MODEL_ID)
    print("[INFO] best_model_dir:", best_model_dir)
    # copy a read-only snapshot of best_model into deploy dir (do not overwrite if exists)
    snapshot_dir = out_dir / "best_model_snapshot"
    if not snapshot_dir.exists():
        try:
            shutil.copytree(best_model_dir, snapshot_dir)
            print("[INFO] snapshot copied to", snapshot_dir)
        except Exception as e:
            print("[WARN] snapshot copy failed:", e)

    # load label_map.json if present
    label_map_path = (ROOT / "data_splits" / LANG / "label_map.json")
    label_map = None
    inv_label_map = None
    if label_map_path.exists():
        try:
            jm = json.load(open(label_map_path, encoding="utf8"))
            label_map = {str(k):int(v) for k,v in jm.get("label_map",{}).items()}
            inv_label_map = {int(v):str(k) for k,v in jm.get("label_map",{}).items()}
            print("[OK] Loaded label_map.json with labels:", list(label_map.keys()))
        except Exception as e:
            print("[WARN] Could not parse label_map.json:", e)
    else:
        print("[WARN] label_map.json not present in splits for", LANG, "- some human-friendly label names may be unavailable.")

    # load tokenizer and model (original student)
    try:
        tokenizer = AutoTokenizer.from_pretrained(str(best_model_dir), use_fast=True)
    except Exception as e:
        print("[ERROR] tokenizer load failed:", e)
        tokenizer = None
    try:
        model = AutoModelForSequenceClassification.from_pretrained(str(best_model_dir)).to(DEVICE).eval()
    except Exception as e:
        print("[ERROR] model load failed:", e)
        model = None

    # Save tokenizer copy to deploy folder
    if tokenizer is not None:
        try:
            tokenizer.save_pretrained(out_dir / "tokenizer")
        except Exception as e:
            print("[WARN] tokenizer.save_pretrained failed:", e)

    # prepare sanity texts for inference (pick few files from dataset folder if available)
    dataset_lang_dir = ROOT / "dataset" / LANG
    sanity_texts = []
    if dataset_lang_dir.exists():
        files = list(dataset_lang_dir.glob("*.txt"))[:BATCH_SIZE_FOR_SANITY]
        for f in files:
            try:
                txt = f.read_text(encoding="utf8", errors="ignore")
                sanity_texts.append(txt[:MAX_TEXT_SANITY])
            except Exception:
                sanity_texts.append("")
    if len(sanity_texts) < BATCH_SIZE_FOR_SANITY:
        # fallback generic text
        sanity_texts += ["Hello world. This is a short sanity test."]*(BATCH_SIZE_FOR_SANITY - len(sanity_texts))
    sanity_texts = sanity_texts[:BATCH_SIZE_FOR_SANITY]

    # create example token tensors for tracing/export
    example_enc = None
    if tokenizer is not None:
        example_enc = tokenizer(sanity_texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
    example_inputs = None
    if example_enc is not None:
        example_inputs = {"input_ids": example_enc["input_ids"], "attention_mask": example_enc["attention_mask"]}

    # record metrics for this LANG
    lang_report = {"language": LANG, "model_dir": str(best_model_dir), "outputs": {}}
    # sizes before
    orig_size = sizeof(best_model_dir)
    lang_report['orig_model_size_bytes'] = int(orig_size)
    lang_report['orig_model_size_human'] = human_bytes(orig_size)

    # 1) TorchScript export (best effort)
    ts_path = out_dir / "torchscript_model.pt"
    ts_ok, ts_msg = False, "skipped_no_example_inputs"
    if model is not None and example_inputs is not None:
        ts_ok, ts_msg = try_torchscript_export(model, example_inputs, ts_path)
        lang_report['outputs']['torchscript'] = {"ok": bool(ts_ok), "message": ts_msg, "path": str(ts_path) if ts_ok else None, "size_bytes": int(ts_path.stat().st_size) if ts_ok and ts_path.exists() else None}
        print("[TORCHSCRIPT] ok:", ts_ok, ts_msg)
    else:
        lang_report['outputs']['torchscript'] = {"ok": False, "message": "no_model_or_example_inputs", "path": None, "size_bytes": None}

    # 2) ONNX export (optional; best effort)
    onnx_path = out_dir / "model.onnx"
    onnx_ok, onnx_msg = False, "skipped"
    if model is not None and tokenizer is not None:
        onnx_ok, onnx_msg = try_onnx_export(model, tokenizer, sanity_texts, onnx_path)
        size = int(onnx_path.stat().st_size) if onnx_ok and onnx_path.exists() else None
        lang_report['outputs']['onnx'] = {"ok": bool(onnx_ok), "message": onnx_msg, "path": str(onnx_path) if onnx_ok else None, "size_bytes": size}
        print("[ONNX] ok:", onnx_ok, onnx_msg)
    else:
        lang_report['outputs']['onnx'] = {"ok": False, "message": "no_model_or_tokenizer", "path": None, "size_bytes": None}

    # 3) Dynamic quantization (CPU) — recommended for transformer Linear-heavy parts
    q_out_dir = out_dir / "quantized_dynamic"
    ensure_dir(q_out_dir)
    quant_ok=False; quant_msg=""
    quant_model_path = q_out_dir / "quantized_model.pth"
    try:
        if model is None:
            quant_ok=False; quant_msg="no_model_loaded"
        else:
            # clone model to cpu
            model_cpu = model.to("cpu").eval()
            # quantize dynamic - choose Linear (and LSTM/GRU if present)
            q_types = {torch.nn.Linear}
            try:
                qmodel = torch.quantization.quantize_dynamic(model_cpu, q_types, dtype=torch.qint8)
                torch.save(qmodel.state_dict(), str(quant_model_path))
                # also save entire quantized module via torch.save (for quick load)
                torch.save(qmodel, str(q_out_dir / "quantized_model_full.pt"))
                quant_ok=True; quant_msg="quantize_dynamic_success"
            except Exception as e:
                quant_ok=False; quant_msg=f"quantize_dynamic_exception:{repr(e)}"
    except Exception as e:
        quant_ok=False; quant_msg=f"quantize_wrapper_exception:{repr(e)}"
    qsize = sizeof(q_out_dir)
    lang_report['outputs']['quantized_dynamic'] = {"ok": bool(quant_ok), "message": quant_msg, "path": str(q_out_dir) if q_out_dir.exists() else None, "size_bytes": int(qsize)}

    # 4) Save tokenizer (already saved earlier), but ensure we store a copy with quantized model for portability
    try:
        if tokenizer is not None:
            tokenizer.save_pretrained(q_out_dir / "tokenizer")
    except Exception as e:
        print("[WARN] saving tokenizer next to quantized model failed:", e)

    # 5) Sanity-run: compare logits/probs for original and quantized where possible
    sanity_results = {}
    try:
        if model is not None and tokenizer is not None:
            # original model inference (on CPU/GPU depending on availability)
            model_device = DEVICE
            model_eval = model.to(model_device).eval()
            enc = tokenizer(sanity_texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
            ids = enc["input_ids"].to(model_device); att = enc["attention_mask"].to(model_device)
            with torch.no_grad():
                out = model_eval(input_ids=ids, attention_mask=att)
                logits_orig = out.logits.detach().cpu().numpy()
                probs_orig = (np.exp(logits_orig) / np.exp(logits_orig).sum(axis=1, keepdims=True))
            sanity_results['original'] = {"logits": logits_orig.tolist(), "probs": probs_orig.tolist()}
        else:
            sanity_results['original'] = None
    except Exception as e:
        sanity_results['original'] = {"error": repr(e)}
        print("[WARN] original model sanity inference failed:", e)

    try:
        q_loaded = None
        if quant_ok:
            # try to load quantized model
            try:
                q_loaded = torch.load(str(q_out_dir / "quantized_model_full.pt"), map_location="cpu")
            except Exception:
                try:
                    # load via state_dict into fresh model
                    q_loaded = AutoModelForSequenceClassification.from_pretrained(str(snapshot_dir)).cpu()
                    q_loaded.load_state_dict(torch.load(str(quant_model_path), map_location="cpu"))
                except Exception as e:
                    q_loaded = None
            # run quantized inference
            if q_loaded is not None and tokenizer is not None:
                q_loaded.eval()
                enc2 = tokenizer(sanity_texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
                ids2 = enc2["input_ids"].cpu(); att2 = enc2["attention_mask"].cpu()
                with torch.no_grad():
                    out_q = q_loaded(input_ids=ids2, attention_mask=att2)
                    logits_q = out_q.logits.detach().cpu().numpy()
                    probs_q = (np.exp(logits_q) / np.exp(logits_q).sum(axis=1, keepdims=True))
                sanity_results['quantized'] = {"logits": logits_q.tolist(), "probs": probs_q.tolist()}
            else:
                sanity_results['quantized'] = None
        else:
            sanity_results['quantized'] = None
    except Exception as e:
        sanity_results['quantized'] = {"error": repr(e)}
        print("[WARN] quantized model sanity inference failed:", e)

    # 6) Latency measurement (original model; small sample)
    latency_ms = None
    try:
        if model is not None and tokenizer is not None:
            # use CPU or GPU depending on model_device
            md = DEVICE
            enc = tokenizer(sanity_texts * 2, padding=True, truncation=True, max_length=256, return_tensors="pt")  # more items
            ids = enc['input_ids'].to(md); att = enc['attention_mask'].to(md)
            # warmup
            with torch.no_grad():
                for _ in range(3): _ = model(input_ids=ids, attention_mask=att)
            t0 = time.time()
            for _ in range(NUM_LATENCY_RUNS):
                with torch.no_grad():
                    _ = model(input_ids=ids, attention_mask=att)
            t1=time.time()
            total_samples = ids.size(0) * NUM_LATENCY_RUNS
            latency_ms = ((t1-t0)/total_samples) * 1000.0
    except Exception as e:
        print("[WARN] latency measurement failed:", e)
        latency_ms = None

    lang_report['latency_ms_per_sample'] = latency_ms
    lang_report['sanity_results'] = sanity_results

    # 7) Save lang_report JSON and metrics
    metrics_path = out_dir / "deploy_metrics.json"
    json.dump(lang_report, open(metrics_path, "w", encoding="utf8"), indent=2)
    print("[SAVED] deploy metrics to", metrics_path)

    # 8) Record overall sizes (orig, ts, onnx, quant)
    sizes = {
        "orig_model_bytes": int(orig_size),
        "torchscript_bytes": int(lang_report['outputs'].get('torchscript', {}).get('size_bytes') or 0),
        "onnx_bytes": int(lang_report['outputs'].get('onnx', {}).get('size_bytes') or 0),
        "quantized_dir_bytes": int(lang_report['outputs'].get('quantized_dynamic', {}).get('size_bytes') or 0)
    }
    json.dump(sizes, open(out_dir / "sizes_summary.json", "w", encoding="utf8"), indent=2)
    print("[SAVED] sizes summary to", out_dir / "sizes_summary.json")
    deploy_report[LANG] = {"deploy_dir": str(out_dir), "metrics_file": str(metrics_path), "sizes_summary": sizes}

# final summary
all_summary_path = DEPLOY_ROOT / "deploy_overview.json"
json.dump(deploy_report, open(all_summary_path, "w", encoding="utf8"), indent=2)
print("\n[ALL DONE] Deploy & quantize complete. Overview saved to:", all_summary_path)


Languages detected: ['English', 'Hindi', 'Marathi']

[LANG] English
[INFO] best_model_dir: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/distillation/English/distil_distilbert-base-multilingual-cased/best_model
[INFO] snapshot copied to /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/deploy_quantize/English/distil_distilbert-base-multilingual-cased/best_model_snapshot
[OK] Loaded label_map.json with labels: ['G', 'PG', 'PG-13', 'R', 'NC-17']


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


[TORCHSCRIPT] ok: True torchscript_trace_success


/tmp/ipython-input-4154277560.py:79: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(


[ONNX] ok: False onnx_failed: UnsupportedOperatorError("Exporting the operator 'aten::scaled_dot_product_attention' to ONNX opset version 13 is not supported. Support for this operator was added in version 14, try exporting with this version")


/tmp/ipython-input-4154277560.py:243: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  qmodel = torch.quantization.quantize_dynamic(model_cpu, q_types, dtype=torch.qint8)
/usr/local/lib/python3.12/dist-packages/torch/_utils.py:444: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only ma

[SAVED] deploy metrics to /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/deploy_quantize/English/distil_distilbert-base-multilingual-cased/deploy_metrics.json
[SAVED] sizes summary to /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/deploy_quantize/English/distil_distilbert-base-multilingual-cased/sizes_summary.json

[LANG] Hindi
[INFO] best_model_dir: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/distillation/Hindi/distil_distilbert-base-multilingual-cased/best_model
[INFO] snapshot copied to /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/deploy_quantize/Hindi/distil_distilbert-base-multilingual-cased/best_model_snapshot
[OK] Loaded label_map.json with labels: ['U', 'UA', 'A']
[TORCHSCRIPT] ok: True torchscript_trace_success
[ONNX] ok: False onnx_failed: UnsupportedOperatorError("Exporting the operator 'aten::scaled_dot_product_attention' to ONNX opset